# 🔥 guato_ba — Métricas e avaliação
### Por que IoU da classe queimada, e o que os modelos entregam
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

A escolha da métrica define o estudo. Como queimada é **rara**, medidas como **acurácia global** ou **mIoU** enganam: um modelo que diz "nada queimou" acerta >99% dos pixels. Por isso a métrica central é o **IoU da classe queimada** — e nunca a acurácia global nem o mIoU.

</div>

# <span style="color:#cf4c39">📏 1. As definições</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Com **TP** (acerto queimada), **FP** (falso positivo) e **FN** (omissão):

- **IoU** = TP / (TP + FP + FN) — a interseção sobre a união da cicatriz (equivale ao CSI/Jaccard).
- **F1 (Dice)** = 2·TP / (2·TP + FP + FN).
- **OE (omissão)** = FN / (TP + FN) = 1 − recall — quanto de queimada **real** o modelo *perdeu*.
- **CE (comissão)** = FP / (TP + FP) = 1 − precisão — quanto do que o modelo *apontou* era **falso**.

IoU alto exige **as duas coisas** ao mesmo tempo: achar a queimada (OE baixo) **sem** inventar queimada (CE baixo).

</div>

In [1]:
def metricas_pixel(TP, FP, FN, TN):
    N = TP+FP+FN+TN; ac = TP+FN; pp = TP+FP
    return dict(
        IoU   = TP/(TP+FP+FN) if (TP+FP+FN) else 0.0,
        F1    = 2*TP/(2*TP+FP+FN) if (2*TP+FP+FN) else 0.0,
        OE    = FN/ac if ac else 0.0,
        CE    = FP/pp if pp else 0.0,
        recall= TP/ac if ac else 0.0,
        OA    = (TP+TN)/N if N else 0.0)

# exemplo: modelo acha quase toda a queimada, mas "pinta demais" (muito FP)
ex = metricas_pixel(TP=800, FP=700, FN=200, TN=1_000_000)
print("Exemplo — recall alto, mas comissão alta:")
for k, v in ex.items(): print(f"  {k:7s} {v:.3f}")
print("\n-> OA=0.999 (parece ótimo!) mas IoU=0.47: a acurácia global ESCONDE o problema.")

Exemplo — recall alto, mas comissão alta:
  IoU     0.471
  F1      0.640
  OE      0.200
  CE      0.467
  recall  0.800
  OA      0.999

-> OA=0.999 (parece ótimo!) mas IoU=0.47: a acurácia global ESCONDE o problema.


# <span style="color:#cf4c39">🗺️ 2. A matriz de confusão da cicatriz</span>
<hr style="border:1px solid #e69627;">

```
                         REFERÊNCIA
                    queimou     não queimou
   MODELO queimou   TP (acerto)  FP (comissão)
   não queimou      FN (omissão) TN (fundo)
```
IoU = TP / (TP + FP + FN) — **o TN fica de fora de propósito** (é o imenso fundo não-queimado). Tudo é calculado **dentro do bioma** e os pixels `255` (inválidos) são descartados.

# <span style="color:#cf4c39">🏆 3. Os resultados do estudo</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

A tabela oficial (`results/artigo/modelos_ml_dl_desempenho.csv`) traz **validação 2024** (ano visto) e **teste OOT 2025** (ano nunca visto). É no OOT que se separa quem generaliza.

</div>

In [2]:
import pandas as pd
df = pd.read_csv(ROOT/"results/artigo/modelos_ml_dl_desempenho.csv") if False else None
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT/"results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
df = pd.read_csv(ROOT/"results/artigo/modelos_ml_dl_desempenho.csv")

cols = ["modelo","tipo","IoU","F1","omissao_OE","comissao_CE"]
print("=== TESTE OOT 2025 (ano nunca visto) — ordenado por IoU ===")
display(df[df.conjunto.str.contains("OOT")].sort_values("IoU", ascending=False)[cols]
          .round(3).reset_index(drop=True))

=== TESTE OOT 2025 (ano nunca visto) — ordenado por IoU ===


,modelo,tipo,IoU,F1,omissao_OE,comissao_CE
0,unet,DL,0.745,0.854,0.222,0.044
1,deeplabv3plus,DL,0.742,0.852,0.224,0.045
2,att_unet,DL,0.730,0.844,0.236,0.046
3,biatt_unet,DL,0.712,0.832,0.266,0.031
4,unet_nored,DL,0.706,0.827,0.265,0.041
5,biunet,DL,0.540,0.702,0.152,0.569
6,rf,ML,0.522,0.686,0.181,0.570
7,xgboost,ML,0.519,0.684,0.179,0.582
8,mlp,ML,0.485,0.653,0.185,0.681
9,svm,ML,0.477,0.646,0.173,0.732


In [3]:
print("=== VALIDAÇÃO 2024 (ano visto) — ordenado por IoU ===")
display(df[df.conjunto.str.contains("validação")].sort_values("IoU", ascending=False)[cols]
          .round(3).reset_index(drop=True))

=== VALIDAÇÃO 2024 (ano visto) — ordenado por IoU ===


,modelo,tipo,IoU,F1,omissao_OE,comissao_CE
0,att_unet,DL,0.979,0.989,0.003,0.018
1,unet,DL,0.976,0.988,0.003,0.022
2,biatt_unet,DL,0.974,0.987,0.003,0.023
3,unet_nored,DL,0.974,0.987,0.004,0.023
4,deeplabv3plus,DL,0.967,0.983,0.004,0.030
5,rf,ML,0.954,0.977,0.003,0.045
6,xgboost,ML,0.952,0.975,0.003,0.048
7,mlp,ML,0.952,0.975,0.003,0.048
8,svm,ML,0.940,0.969,0.002,0.062
9,resnet34_siam,DL,0.857,0.923,0.000,0.166


<div style="text-align: justify; margin-right: 2.5%;">

**O que ler aqui:** em 2024 (visto) quase todos acertam >0,95 de IoU. O que importa é a **queda** para 2025: as melhores U-Nets seguram IoU ~0,74, enquanto os modelos por pixel caem para ~0,5 (comissão dispara). A **diferença val→teste** é a medida honesta de generalização.

</div>

# <span style="color:#cf4c39">🌎 4. Do patch ao produto: acumulado vs. referências externas</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

No produto anual (notebook 05), o acumulado é comparado às referências independentes **HLS**, **DETER** e **MapBiomas**, sempre reportando a **% do Pantanal efetivamente analisada**.

</div>

In [4]:
ref_csv = ROOT/"results/artigo/acumulado_2024_vs_referencias.csv"
if ref_csv.exists():
    r = pd.read_csv(ref_csv)
    display(r[["modelo","dominio","referencia","cobertura_pct","IoU","OE","CE","area_modelo_ha","area_ref_ha"]]
              .head(6).round(3))
else:
    print("(tabela de referências não encontrada neste checkout)")

,modelo,dominio,referencia,cobertura_pct,IoU,OE,CE,area_modelo_ha,area_ref_ha
0,unet,pantanal_inteiro,referencia,98.325,0.445,0.123,0.971,2552303.84,1380939.08
1,unet,pantanal_inteiro,DETER,98.325,0.451,0.325,0.498,2552303.84,2176502.68
2,unet,pantanal_inteiro,MapBiomas,98.325,0.476,0.307,0.455,2552303.84,2223326.04
3,unet,analisado_SCL,referencia,98.325,0.448,0.112,0.984,2552300.60,1363689.24
4,unet,analisado_SCL,DETER,98.325,0.457,0.311,0.508,2552300.60,2130963.24
5,unet,analisado_SCL,MapBiomas,98.325,0.483,0.291,0.466,2552300.60,2172866.80


---
Continue no **`05_acumulado_anual`** para ver como nasce o mapa do Pantanal inteiro.